In [1]:
import os

os.chdir('..')
os.getcwd()

'c:\\Users\\chouybat\\repositories\\vfm-quality-control-benchmark'

In [4]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact_manual
from PIL import Image

from vfm_quality_control_benchmark.data_types.enums import ModelName
from vfm_quality_control_benchmark.models.factory import Factory
from vfm_quality_control_benchmark.utils.tasks import to_anomaly_map

config = Factory.load_config(
    ModelName.SAM3, Path('src/vfm_quality_control_benchmark/configs/models.yaml')
)
model = Factory.build_model(config=config)

[transformers] `memory_attention_rope_theta` is deprecated and will be removed in v5.0. Use `rope_parameters['rope_theta']` instead.


Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

In [ ]:
# image: path relative to repo root; prompt: comma-separated
@interact_manual(
    image='data/3CAD/3CAD/Aluminum_Camera_Cover/test/scratche/000001.png',
    prompt='scratch',
)
def show(image: str, prompt: str) -> None:
    path = Path(image)
    img = Image.open(path).convert('RGB')
    pred = model.predict(img, [p.strip() for p in prompt.split(',')])
    amap, score = to_anomaly_map(pred)
    gt = path.parents[2] / 'ground_truth' / path.parent.name / path.name

    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    for a in ax:
        a.imshow(img)
        a.axis('off')
        if gt.exists():
            a.contour(np.array(Image.open(gt)) > 0, colors='lime')
    ax[1].imshow(amap, cmap='jet', alpha=0.5, vmin=0, vmax=1)
    ax[1].set_title(f'image score {score:.2f}')
    plt.show()

interactive(children=(Text(value='data/3CAD/3CAD/Aluminum_Camera_Cover/test/scratche/000001.png', continuous_u…